<a href="https://colab.research.google.com/github/ultimog/The-Cost-of-Living-Brief/blob/main/ps3_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. Write the SQL yourself; that is the skill being assessed and it is the one interviewers test. AI is authorised and required in Phase 4.

**Note on AI use:** AI assistance was used in Phases 1–3 under the updated course policy for this problem set (announced [date / where]).

## Setup

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [21]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 3_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

In [22]:
N = N_CUSTOMERS

# About 70% of customers are in the West, so the West has at least 2,000.
# p= gives the chance of each region
region = rng.choice(["West", "East", "South"], size=N, p=[0.70, 0.15, 0.15])

# Order counts: 1 + NegativeBinomial(0.8, 0.12), the distribution the brief requires
n_orders = 1 + rng.negative_binomial(NB_N, NB_P, size=N)

# Revenue depends on order count; np.where sets anything below the floor to 50
raw_revenue = REV_PER_ORDER * n_orders + rng.normal(REV_MEAN, REV_SD, size=N)
revenue = np.where(raw_revenue < REV_FLOOR, REV_FLOOR, raw_revenue)

# Signup dates: a random number of days after 1 January 2022
signup_date = pd.Timestamp("2022-01-01") + pd.to_timedelta(rng.integers(0, 1095, size=N), unit="D")

customers = pd.DataFrame({"customer_id": np.arange(1, N + 1),
                          "region": region,
                          "revenue": revenue.round(2),
                          "signup_date": signup_date})

# One row per order: np.repeat writes each customer's id once per order, as in Lab 11 Step 2
orders = pd.DataFrame({"order_id": np.arange(1, n_orders.sum() + 1),
                       "customer_id": np.repeat(customers["customer_id"].values, n_orders)})

assert (region == "West").sum() >= MIN_WEST, "fewer than 2,000 West customers"

# DuckDB reads the DataFrames customers and orders by their Python names, so they are registered as soon as they exist. Confirm it:
print(con.execute("SELECT COUNT(*) AS customers FROM customers").df())
print(con.execute("SELECT COUNT(*) AS orders FROM orders").df())
print(con.execute("SELECT region, COUNT(*) AS n FROM customers GROUP BY region ORDER BY region").df())
print(f"Mean orders per customer: {n_orders.mean():.2f}, largest: {n_orders.max()}")


   customers
0       3000
   orders
0   20584
  region     n
0   East   443
1  South   460
2   West  2097
Mean orders per customer: 6.86, largest: 57


I made 3,000 customers (seed 42), and 2,097 of them ended up in the West. Together they placed 20,584 orders. Most customers only order a handful of times, but a few order a lot (one placed 57). I also made revenue go up with the number of orders, on purpose, because that's the whole reason the two queries end up disagreeing.

### Step 1.2: Write both queries (7 pts)

In [23]:
# The low number: one row per customer
customer_level = con.execute("""
    SELECT AVG(revenue) AS avg_revenue, COUNT(*) AS rows_used
    FROM customers
    WHERE region = 'West'
""").df()

# The high number: joining orders repeats each customer once per order
order_level = con.execute("""
    SELECT AVG(c.revenue) AS avg_revenue, COUNT(*) AS rows_used
    FROM customers AS c
    JOIN orders AS o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
""").df()

low, high = customer_level["avg_revenue"][0], order_level["avg_revenue"][0]
print(f"Un-joined: ${low:,.2f} over {customer_level['rows_used'][0]:,} rows")
print(f"Joined:    ${high:,.2f} over {order_level['rows_used'][0]:,} rows")
print(f"Ratio:     {high / low:.2f}")

Un-joined: $814.43 over 2,097 rows
Joined:    $1,250.67 over 14,191 rows
Ratio:     1.54


> **🛑 Verification checkpoint.** On West-region customers you should get about
> **$812** from the un-joined query (accept $780 to $845) and about **$1,240**
> from the joined one (accept $1,160 to $1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

The query without the join gives \$814.43, using 2,097 rows (one per West customer). The query with the join gives \$1,250.67, using 14,191 rows (one per West order). The joined number is 1.54x bigger.

The right number is \$814. The question asks for average revenue per customer, so each customer should count once. Once you join in the orders table, though, every customer shows up once for each order they placed. The customers who order the most also bring in the most revenue, so they get counted over and over, and that drags the average up. So the \$1,240 isn't really an average over customers. It's an average over orders that just happens to use a customer column.

### Step 1.3: The rule (10 pts)

In [24]:
def check_join(con, before_sql, after_sql):
    """Warn if a join changes the number of rows. Returns True if the row count is unchanged."""
    # WITH names a query so we can count its rows
    n_before = con.execute(f"WITH q AS ({before_sql}) SELECT COUNT(*) AS n FROM q").df()["n"][0]
    n_after = con.execute(f"WITH q AS ({after_sql}) SELECT COUNT(*) AS n FROM q").df()["n"][0]
    if n_after != n_before:
        print(f"WARNING: the join changed the row count from {n_before:,} to {n_after:,} "
              f"({n_after / n_before:.1f}x). Each row may be counted more than once.")
        return False
    print(f"OK: row count unchanged ({n_before:,} rows).")
    return True


west_customers = "SELECT * FROM customers WHERE region = 'West'"

# The bad query, before averaging: West customers joined to their orders
fanned_out = """
    SELECT c.* FROM customers AS c
    JOIN orders AS o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

# A safe join: orders counted per customer FIRST, so there's one row per customer to join to
safe_join = """
    WITH order_counts AS (
        SELECT customer_id, COUNT(*) AS n_orders FROM orders GROUP BY customer_id
    )
    SELECT c.*, o.n_orders FROM customers AS c
    JOIN order_counts AS o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

print("Bad query:");  _ = check_join(con, west_customers, fanned_out)
print("Good query:"); _ = check_join(con, west_customers, west_customers)
print("Safe join:");  _ = check_join(con, west_customers, safe_join)


Bad query:
Good query:
OK: row count unchanged (2,097 rows).
Safe join:
OK: row count unchanged (2,097 rows).


My check counts the rows before and after a join. On the bad query it gives a. warning: the join turns 2,097 West customers into 14,191 rows, almost 7 times as many, so customers are clearly being double counted. On the good query there's no join, so it stays quiet. It also stays quiet when I join to a table that already has one row per customer (orders counted up per customer first). So the rule isn't "never join." It's "a join shouldn't change what one row means." If this check ran on every query behind a dashboard, the \$1,240 would have been caught before it ever got near a slide.

## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [25]:
# About 15% of customers get a missing region
is_missing = rng.random(N) < 0.15
customers.loc[is_missing, "region"] = None      # None is stored as NULL when DuckDB reads it

print(con.execute("SELECT COUNT(*) AS missing_region FROM customers WHERE region IS NULL").df())


   missing_region
0             401


I wiped out the region for about 15% of customers to represent people who signed up before the region field was required. That left 401 customers with no region.

### Step 2.2: Lose them, then find them (15 pts)

In [26]:
lost = con.execute("SELECT COUNT(*) AS n FROM customers WHERE region != 'West'").df()["n"][0]
found = con.execute("""
    SELECT COUNT(*) AS n FROM customers
    WHERE region != 'West' OR region IS NULL
""").df()["n"][0]

print(f"WHERE region != 'West':                  {lost:,} rows")
print(f"WHERE region != 'West' OR region IS NULL: {found:,} rows")
print(f"Customers silently dropped:               {found - lost:,}")


WHERE region != 'West':                  777 rows
WHERE region != 'West' OR region IS NULL: 1,178 rows
Customers silently dropped:               401


`WHERE region != 'West'` gives back 777 customers. But what the analyst actually wanted was everyone outside the West, which is 1,178. The 401 missing are exactly the customers with no region.

Why it happens: if a customer's region is blank, SQL can't tell whether it's 'West' or not, so the comparison comes out as "unknown" instead of true or false, and `WHERE` only keeps rows where the answer is true.

Every query that filters or groups by region quietly skips these customers. They could potentially be different from everyone else, such as newer, smaller, or more likely to leave, but removing them makes every regional number only describe the older, more settled customers. Each region looks healthier than it is; the regional totals won't add up to the company total. And because the dropped customers never show up anywhere in the output, nobody would know to look for them.

### Step 2.3: Aggregate without collapsing (10 pts)

In [27]:
with_region_avg = con.execute("""
    SELECT customer_id,
           region,
           revenue,
           AVG(revenue) OVER (PARTITION BY region)           AS region_avg,
           revenue - AVG(revenue) OVER (PARTITION BY region) AS above_region_avg
    FROM customers
    ORDER BY customer_id
""").df()

print(f"Rows in customers: {N:,}   rows in the result: {len(with_region_avg):,}")
with_region_avg.head(10)

Rows in customers: 3,000   rows in the result: 3,000


,customer_id,region,revenue,region_avg,above_region_avg
0,1,East,1091.60,822.734,268.866
1,2,West,791.79,809.840,-18.050
2,3,South,803.03,804.398,-1.368
3,4,West,503.94,809.840,-305.900
4,5,West,1411.17,809.840,601.330
5,6,South,381.69,804.398,-422.708
6,7,East,2738.42,822.734,1915.686
7,8,East,554.64,822.734,-268.094
8,9,West,1725.53,809.840,915.690
9,10,West,1167.00,809.840,357.160


This puts each customer's revenue right next to their region's average in a single query: 3,000 rows in, 3,000 rows out, no join and no subquery. GROUP BY would squash each region down to one row, but the window function keeps every customer and just adds their region's average beside them, so the director can see who's above and who's below. The 401 customers with no region all get lumped together, so their region average isn't really a region.

## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [28]:
import statsmodels.api as sm      # for sm.add_constant and sm.OLS, as in Lab 10

def coef_on_d(y, d, controls):
    """Regress y on d plus every array in controls, with a constant; return d's coefficient."""
    X = sm.add_constant(np.column_stack([d] + controls))
    return sm.OLS(y, X).fit().params[1]


n = 10_000
TRUE_EFFECT = 50

# Existing account size drives BOTH which customers are in the West and their revenue
account_size = rng.normal(0, 1, n)
in_west = (rng.random(n) < 1 / (1 + np.exp(-1.5 * account_size))).astype(float)   # bigger accounts more likely West
programme_revenue = 800 + TRUE_EFFECT * in_west + 300 * account_size + rng.normal(0, 150, n)

naive = coef_on_d(programme_revenue, in_west, [])
adjusted = coef_on_d(programme_revenue, in_west, [account_size])

print(f"Share of customers in the West: {in_west.mean():.1%}")
print(f"{'estimate':<34}{'value':>8}{'truth':>8}")
print(f"{'Naive (West minus other regions)':<34}{naive:>8.1f}{TRUE_EFFECT:>8}")
print(f"{'Adjusted for account size':<34}{adjusted:>8.1f}{TRUE_EFFECT:>8}")

Share of customers in the West: 49.4%
estimate                             value   truth
Naive (West minus other regions)     373.7      50
Adjusted for account size             47.1      50


In this simulated world, I know the programme's real effect is \$50 per customer. Comparing the west with all the other regions, makes the programme look like it's worth \$374 per customer, 7x the real effect. That's because bigger accounts were more likely to end up in the West, and bigger accounts bring in more money whether or not they got the programme. Once I control for account size, the estimate drops to \$47, very close to the real \$50.

### Step 3.2: The control that ruins it (13 pts)

In [29]:
# Support tickets are caused by BOTH the programme and revenue
tickets = 3 * in_west + 0.01 * programme_revenue + rng.normal(0, 1, n)

with_tickets = coef_on_d(programme_revenue, in_west, [account_size, tickets])

print(f"{'estimate':<40}{'value':>8}{'truth':>8}")
print(f"{'Adjusted for account size':<40}{adjusted:>8.1f}{TRUE_EFFECT:>8}")
print(f"{'Adjusted for account size + tickets':<40}{with_tickets:>8.1f}{TRUE_EFFECT:>8}")

estimate                                   value   truth
Adjusted for account size                   47.1      50
Adjusted for account size + tickets       -194.3      50


Adding support tickets as a control moved the estimate from \$47(close to the real \$50) all the way to −\$194.

**To the director:** Support tickets might sound viable to measure engagement, but it doesn't work here. The programme itself creates tickets, since account managers push customers to reach out, and customers who spend more also file more tickets. So when we compare customers who have the same number of tickets, we're really comparing two different kinds of people: a programme customer, who got some of those tickets from the programme, and a non-programme customer with the same count, who must have gotten them by spending more. That means at every ticket level, the non-programme customers we're comparing against are the bigger spenders, and the programme looks worse than it really is.

## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The query reviewer (12 pts)

In [ ]:
def review_query(con, sql):
    """The model's function from the P.R.I.M.E. prompt, then your corrections:
    returns a list of structured warnings for one SELECT."""
    ...

In [33]:

import re
import pandas as pd


def review_query(con, sql):
    """
    Review a SELECT query for:
      1. Join cardinality changes / fan-out.
      2. Nullable-column filters that discard NULL rows.
      3. Aggregates potentially inflated by join fan-out.

    Returns a list of structured warning dictionaries.

    Supports simple equality joins and single-table references.
    """

    warnings = []

    def warn(code, severity, message, **details):
        warnings.append({
            "code": code,
            "severity": severity,
            "message": message,
            "details": details
        })

    sql = sql.strip().rstrip(";")

    if not re.match(r"^\s*(WITH\b.*)?SELECT\b",
                    sql, re.IGNORECASE | re.DOTALL):
        raise ValueError("Only SELECT queries are supported.")

    # Identify FROM/JOIN tables and their aliases.
    table_pattern = re.compile(
        r"\b(FROM|JOIN)\s+"
        r'("?[a-zA-Z_][\w.]*"?)'
        r"(?:\s+(?:AS\s+)?([a-zA-Z_]\w*))?",
        re.IGNORECASE
    )

    tables = {}

    for match in table_pattern.finditer(sql):
        table = match.group(2).replace('"', "")
        alias = match.group(3)

        if alias and alias.upper() in {
            "WHERE", "JOIN", "ON", "LEFT", "RIGHT",
            "INNER", "FULL", "CROSS", "GROUP", "ORDER"
        }:
            alias = None

        tables[alias or table.split(".")[-1]] = table

    if not tables:
        return warnings

    def quote_identifier(identifier):
        return ".".join(
            '"' + part.replace('"', '""') + '"'
            for part in identifier.split(".")
        )

    def table_info(table):
        return con.execute(
            f"DESCRIBE SELECT * FROM {quote_identifier(table)}"
        ).df()

    def has_nulls(table, column):
        result = con.execute(
            f"""
            SELECT COUNT(*)
            FROM {quote_identifier(table)}
            WHERE {quote_identifier(column)} IS NULL
            """
        ).fetchone()[0]
        return result > 0

    # ---------------------------------------------------------
    # 1. Detect join cardinality changes.
    # ---------------------------------------------------------

    join_pattern = re.compile(
        r"\b(LEFT|RIGHT|INNER|FULL)?\s*JOIN\s+"
        r'("?[a-zA-Z_][\w.]*"?)'
        r"(?:\s+(?:AS\s+)?([a-zA-Z_]\w*))?"
        r"\s+ON\s+"
        r'([a-zA-Z_]\w*)\.([a-zA-Z_]\w*)'
        r"\s*=\s*"
        r'([a-zA-Z_]\w*)\.([a-zA-Z_]\w*)',
        re.IGNORECASE
    )

    joins = []

    for match in join_pattern.finditer(sql):
        join_type = (match.group(1) or "INNER").upper()
        join_table = match.group(2).replace('"', "")
        join_alias = match.group(3) or join_table.split(".")[-1]

        alias1, col1 = match.group(4), match.group(5)
        alias2, col2 = match.group(6), match.group(7)

        if alias1 not in tables or alias2 not in tables:
            continue

        left_table = tables[alias1]
        right_table = tables[alias2]

        joins.append({
            "type": join_type,
            "left_table": left_table,
            "left_key": col1,
            "right_table": right_table,
            "right_key": col2,
        })

    fanout_detected = False

    for join in joins:
        right_table = join["right_table"]
        right_key = join["right_key"]

        # Check uniqueness on the lookup side WITHOUT executing
        # the complete joined query a second time.
        #
        # GROUP BY checks the join key directly on the underlying
        # table. Duplicate keys mean one left-side row can match
        # multiple right-side rows.
        duplicates = con.execute(
            f"""
            SELECT {quote_identifier(right_key)}, COUNT(*) AS n
            FROM {quote_identifier(right_table)}
            WHERE {quote_identifier(right_key)} IS NOT NULL
            GROUP BY {quote_identifier(right_key)}
            HAVING COUNT(*) > 1
            LIMIT 5
            """
        ).df()

        if not duplicates.empty:
            fanout_detected = True

            warn(
                "JOIN_FANOUT",
                "error",
                "Join key is not unique on the right-hand side. "
                "Rows from the left table may be multiplied.",
                left_table=join["left_table"],
                right_table=right_table,
                join_key=right_key,
                sample_duplicate_keys=duplicates.to_dict(
                    orient="records"
                ),
                join_type=join["type"]
            )

        # An INNER JOIN can also remove rows that have no match.
        if join["type"] == "INNER":
            left_table = join["left_table"]
            left_key = join["left_key"]

            unmatched = con.execute(
                f"""
                SELECT COUNT(*)
                FROM {quote_identifier(left_table)} l
                WHERE l.{quote_identifier(left_key)} IS NOT NULL
                  AND NOT EXISTS (
                      SELECT 1
                      FROM {quote_identifier(right_table)} r
                      WHERE r.{quote_identifier(right_key)}
                            = l.{quote_identifier(left_key)}
                  )
                """
            ).fetchone()[0]

            if unmatched:
                warn(
                    "JOIN_ROW_LOSS",
                    "warning",
                    "INNER JOIN removes left-side rows without matches.",
                    left_table=left_table,
                    right_table=right_table,
                    unmatched_left_rows=unmatched
                )

    # ---------------------------------------------------------
    # 2. Detect filters on nullable columns.
    # ---------------------------------------------------------

    where_match = re.search(
        r"\bWHERE\b(.*?)(?:\bGROUP\s+BY\b|\bORDER\s+BY\b|"
        r"\bHAVING\b|\bLIMIT\b|$)",
        sql,
        re.IGNORECASE | re.DOTALL
    )

    if where_match:
        where_clause = where_match.group(1)

        for alias, table in tables.items():
            info = table_info(table)

            for _, column_info in info.iterrows():
                column = column_info["column_name"]

                # Match qualified or unqualified column references
                # in ordinary comparison predicates.
                predicate = re.search(
                    rf"(?<![\w.])"
                    rf"(?:{re.escape(alias)}\s*\.\s*)?"
                    rf"{re.escape(column)}"
                    rf"\s*(?:=|<>|!=|<|>|<=|>=|LIKE|IN\s*\()",
                    where_clause,
                    re.IGNORECASE
                )

                if not predicate:
                    continue

                # DuckDB's actual data is checked as well as its
                # schema. This catches nullable data even when
                # schema metadata does not explicitly describe it.
                nullable = (
                    str(column_info.get("null", "")).upper()
                    == "YES"
                )

                if nullable or has_nulls(table, column):
                    if has_nulls(table, column):
                        warn(
                            "NULL_FILTER",
                            "warning",
                            "WHERE predicate filters a column containing "
                            "NULL values. SQL comparisons evaluate to "
                            "UNKNOWN for NULL, so those rows are excluded.",
                            table=table,
                            column=column,
                            alias=alias,
                            predicate=predicate.group(0).strip()
                        )

    # ---------------------------------------------------------
    # 3. Detect aggregates exposed to fan-out.
    # ---------------------------------------------------------

    has_aggregate = bool(re.search(
        r"\b(SUM|AVG|COUNT|MIN|MAX)\s*\(",
        sql,
        re.IGNORECASE
    ))

    if has_aggregate and fanout_detected:
        warn(
            "AGGREGATE_FANOUT",
            "error",
            "An aggregate is calculated over a result containing "
            "a potentially multiplying join. Measures may be "
            "overstated.",
            aggregates=re.findall(
                r"\b(SUM|AVG|COUNT|MIN|MAX)\s*\(",
                sql,
                re.IGNORECASE
            ),
            recommendation=(
                "Aggregate at the intended grain before joining, "
                "or verify the join cardinality."
            )
        )

    return warnings


In [34]:
# My Phase 1 and Phase 2 queries, plus two tests of the reviewer's limits
queries = {
    "Phase 1, un-joined ($812)": """
        SELECT AVG(revenue) AS avg_revenue FROM customers WHERE region = 'West'
    """,
    "Phase 1, joined ($1,240)": """
        SELECT AVG(c.revenue) AS avg_revenue
        FROM customers AS c
        JOIN orders AS o ON c.customer_id = o.customer_id
        WHERE c.region = 'West'
    """,
    "Phase 2, != filter (drops NULLs)": """
        SELECT COUNT(*) AS n FROM customers WHERE region != 'West'
    """,
    "Phase 2, fixed filter": """
        SELECT COUNT(*) AS n FROM customers WHERE region != 'West' OR region IS NULL
    """,
    "Test: same bad join, ON written the other way round": """
        SELECT AVG(c.revenue) AS avg_revenue
        FROM customers AS c
        JOIN orders AS o ON o.customer_id = c.customer_id
        WHERE c.region = 'West'
    """,
    "Test: safe join from Step 1.3 (uses WITH)": safe_join,
}

for name, sql in queries.items():
    print(name)
    try:
        warnings = review_query(con, sql)
    except Exception as err:
        print("   the reviewer crashed:", str(err).splitlines()[0])
        print()
        continue
    if len(warnings) == 0:
        print("   no warnings")
    for w in warnings:
        print("   -", w["code"], ":", w["message"])
    print()

Phase 1, un-joined ($812)
   - NULL_FILTER : WHERE predicate filters a column containing NULL values. SQL comparisons evaluate to UNKNOWN for NULL, so those rows are excluded.

Phase 1, joined ($1,240)
   - JOIN_FANOUT : Join key is not unique on the right-hand side. Rows from the left table may be multiplied.
   - NULL_FILTER : WHERE predicate filters a column containing NULL values. SQL comparisons evaluate to UNKNOWN for NULL, so those rows are excluded.
   - AGGREGATE_FANOUT : An aggregate is calculated over a result containing a potentially multiplying join. Measures may be overstated.

Phase 2, != filter (drops NULLs)
   - NULL_FILTER : WHERE predicate filters a column containing NULL values. SQL comparisons evaluate to UNKNOWN for NULL, so those rows are excluded.

Phase 2, fixed filter
   - NULL_FILTER : WHERE predicate filters a column containing NULL values. SQL comparisons evaluate to UNKNOWN for NULL, so those rows are excluded.

Test: same bad join, ON written the other wa

**What it caught:** On the joined Phase 1 query, it flagged the fan-out (`JOIN_FANOUT`) because customer_id repeats in the orders table, and flagged the average as computed over a fanned-out result (`AGGREGATE_FANOUT`). On the Phase 2 `!=` query, it flagged the null filter (`NULL_FILTER`), because region contains NULLs that the comparison silently drops.

**What it missed:**
- It gave false alarms on two correct queries. It flagged the un-joined \$812 query, where dropping customers with no region is correct because I only want the West, and it flagged my fixed Phase 2 query even though `OR region IS NULL` keeps the missing regions. It only checks whether a compared column has NULLs; it doesn't check whether the query already handles them.
- It never actually counts rows before and after a join. It only checks whether the join key repeats in the table written after the `=` in the ON line. When I wrote the same bad join with the ON the other way round (`o.customer_id = c.customer_id`), it gave no fan-out warning at all, even though the query still returns the inflated average.
- It crashed on my safe join from Step 1.3, because it treats the name defined in the WITH block (`order_counts`) as a real table.

**Would it have caught the \$1,240?** Yes, as the query was written, because customer_id repeats in orders, so the fan-out warning fires. But only because of the order the ON line happened to be written in: the same query with the two sides swapped passes with no fan-out warning. My Step 1.3 check, which counts the rows before and after the join, catches it either way.

**My corrections to its code:** None. I ran the function exactly as ChatGPT wrote it.

### Task 4.2: The one-page writeup (8 pts)

The right number is \$814.** The \$1,240 came from a query that joined customers to their orders before averaging, so each customer got counted once per order. Big customers place more orders, so they got counted many times, and that pushed the average up by about 54%. The reason the numbers differed is that one query quietly changed what a row meant, from a customer to an order, and nothing caught it.

The programme works, but not as much as the regional gap makes it look. In a simulation where I know the real effect is \$50 per customer, comparing the West to other regions gives \$374, because bigger accounts were more likely to be in the West. Adjusting for account size gives \$47. The addition of support tickets as a control only made the effect look worse at −\$194, because the programme itself changes how many tickets people file. To really know, we'd need to roll the programme out to randomly chosen customers, or at least compare customers of similar size.

**What I'd recommend before any number goes on a slide:**
1. Check that joins don't multiply rows when we're averaging per customer.
2. Make sure queries keep customers with a missing region instead of silently dropping some of them.
3. Have a second analyst rerun any number before it goes to the board.

### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

**Model and date:** GPT-4o mini 1/1/26

**Prompt:**


[Prep] Act as an analytics engineer who reviews SQL before it reaches
a dashboard.

[Request] Write a Python function review_query(con, sql) that runs
against DuckDB and returns a list of warnings for a given SELECT. It
should flag at minimum: a join that changes the row count, a filter on
a nullable column that will silently drop unknowns, and an aggregate
computed over a fanned-out result.

[Iterate] duckdb and pandas only. Reuse my table names, customers and
orders. Return structured warnings, not printed text, so I can test it.

[Mechanism Check] Add comments explaining how you detect fan-out
without running the full join twice, and why three-valued logic makes
the null filter a correctness issue rather than a style issue.

[Evaluate] Run it against both of my Phase 1 queries and tell me
whether it would have caught the $1,240 before it reached the board.
```
The key distinction is between join fan-out risk and a confirmed incorrect metric. A join can multiply rows without producing an obvious SQL error, and an aggregate such as `SUM(orders.amount)` can then report a plausible but inflated number.

I'll make `review_query(con, sql)` return structured dictionaries that you can assert against in tests. The implementation will use DuckDB for metadata and cardinality checks, pandas only where useful, and no additional SQL parsing libraries.

One important limitation: I don't have your two Phase 1 SQL statements or the data in your DuckDB connection in this conversation. I'll provide the reviewer first, then explain exactly what I need to verify the $1,240 incident rather than inventing a result.

## 1. Implementation

This version targets the common analytics pattern:

- `customers` joined to `orders`.
- Equality joins such as `customers.customer_id = orders.customer_id`.
- Optional aliases.
- Standard `WHERE` predicates.
- Aggregates such as `SUM`, `COUNT`, and `AVG`.

It deliberately reports unsupported or ambiguous patterns conservatively rather than pretending to understand arbitrary SQL.

python

```python

import re
import pandas as pd


def review_query(con, sql):
    """
    Review a SELECT query for:
      1. Join cardinality changes / fan-out.
      2. Nullable-column filters that discard NULL rows.
      3. Aggregates potentially inflated by join fan-out.

    Returns a list of structured warning dictionaries.

    Supports simple equality joins and single-table references.
    """

    warnings = []

    def warn(code, severity, message, **details):
        warnings.append({
            "code": code,
            "severity": severity,
            "message": message,
            "details": details
        })

    sql = sql.strip().rstrip(";")

    if not re.match(r"^\s*(WITH\b.*)?SELECT\b",
                    sql, re.IGNORECASE | re.DOTALL):
        raise ValueError("Only SELECT queries are supported.")

    # Identify FROM/JOIN tables and their aliases.
    table_pattern = re.compile(
        r"\b(FROM|JOIN)\s+"
        r'("?[a-zA-Z_][\w.]*"?)'
        r"(?:\s+(?:AS\s+)?([a-zA-Z_]\w*))?",
        re.IGNORECASE
    )

    tables = {}

    for match in table_pattern.finditer(sql):
        table = match.group(2).replace('"', "")
        alias = match.group(3)

        if alias and alias.upper() in {
            "WHERE", "JOIN", "ON", "LEFT", "RIGHT",
            "INNER", "FULL", "CROSS", "GROUP", "ORDER"
        }:
            alias = None

        tables[alias or table.split(".")[-1]] = table

    if not tables:
        return warnings

    def quote_identifier(identifier):
        return ".".join(
            '"' + part.replace('"', '""') + '"'
            for part in identifier.split(".")
        )

    def table_info(table):
        return con.execute(
            f"DESCRIBE SELECT * FROM {quote_identifier(table)}"
        ).df()

    def has_nulls(table, column):
        result = con.execute(
            f"""
            SELECT COUNT(*)
            FROM {quote_identifier(table)}
            WHERE {quote_identifier(column)} IS NULL
            """
        ).fetchone()[0]
        return result > 0

    # ---------------------------------------------------------
    # 1. Detect join cardinality changes.
    # ---------------------------------------------------------

    join_pattern = re.compile(
        r"\b(LEFT|RIGHT|INNER|FULL)?\s*JOIN\s+"
        r'("?[a-zA-Z_][\w.]*"?)'
        r"(?:\s+(?:AS\s+)?([a-zA-Z_]\w*))?"
        r"\s+ON\s+"
        r'([a-zA-Z_]\w*)\.([a-zA-Z_]\w*)'
        r"\s*=\s*"
        r'([a-zA-Z_]\w*)\.([a-zA-Z_]\w*)',
        re.IGNORECASE
    )

    joins = []

    for match in join_pattern.finditer(sql):
        join_type = (match.group(1) or "INNER").upper()
        join_table = match.group(2).replace('"', "")
        join_alias = match.group(3) or join_table.split(".")[-1]

        alias1, col1 = match.group(4), match.group(5)
        alias2, col2 = match.group(6), match.group(7)

        if alias1 not in tables or alias2 not in tables:
            continue

        left_table = tables[alias1]
        right_table = tables[alias2]

        joins.append({
            "type": join_type,
            "left_table": left_table,
            "left_key": col1,
            "right_table": right_table,
            "right_key": col2,
        })

    fanout_detected = False

    for join in joins:
        right_table = join["right_table"]
        right_key = join["right_key"]

        # Check uniqueness on the lookup side WITHOUT executing
        # the complete joined query a second time.
        #
        # GROUP BY checks the join key directly on the underlying
        # table. Duplicate keys mean one left-side row can match
        # multiple right-side rows.
        duplicates = con.execute(
            f"""
            SELECT {quote_identifier(right_key)}, COUNT(*) AS n
            FROM {quote_identifier(right_table)}
            WHERE {quote_identifier(right_key)} IS NOT NULL
            GROUP BY {quote_identifier(right_key)}
            HAVING COUNT(*) > 1
            LIMIT 5
            """
        ).df()

        if not duplicates.empty:
            fanout_detected = True

            warn(
                "JOIN_FANOUT",
                "error",
                "Join key is not unique on the right-hand side. "
                "Rows from the left table may be multiplied.",
                left_table=join["left_table"],
                right_table=right_table,
                join_key=right_key,
                sample_duplicate_keys=duplicates.to_dict(
                    orient="records"
                ),
                join_type=join["type"]
            )

        # An INNER JOIN can also remove rows that have no match.
        if join["type"] == "INNER":
            left_table = join["left_table"]
            left_key = join["left_key"]

            unmatched = con.execute(
                f"""
                SELECT COUNT(*)
                FROM {quote_identifier(left_table)} l
                WHERE l.{quote_identifier(left_key)} IS NOT NULL
                  AND NOT EXISTS (
                      SELECT 1
                      FROM {quote_identifier(right_table)} r
                      WHERE r.{quote_identifier(right_key)}
                            = l.{quote_identifier(left_key)}
                  )
                """
            ).fetchone()[0]

            if unmatched:
                warn(
                    "JOIN_ROW_LOSS",
                    "warning",
                    "INNER JOIN removes left-side rows without matches.",
                    left_table=left_table,
                    right_table=right_table,
                    unmatched_left_rows=unmatched
                )

    # ---------------------------------------------------------
    # 2. Detect filters on nullable columns.
    # ---------------------------------------------------------

    where_match = re.search(
        r"\bWHERE\b(.*?)(?:\bGROUP\s+BY\b|\bORDER\s+BY\b|"
        r"\bHAVING\b|\bLIMIT\b|$)",
        sql,
        re.IGNORECASE | re.DOTALL
    )

    if where_match:
        where_clause = where_match.group(1)

        for alias, table in tables.items():
            info = table_info(table)

            for _, column_info in info.iterrows():
                column = column_info["column_name"]

                # Match qualified or unqualified column references
                # in ordinary comparison predicates.
                predicate = re.search(
                    rf"(?<![\w.])"
                    rf"(?:{re.escape(alias)}\s*\.\s*)?"
                    rf"{re.escape(column)}"
                    rf"\s*(?:=|<>|!=|<|>|<=|>=|LIKE|IN\s*\()",
                    where_clause,
                    re.IGNORECASE
                )

                if not predicate:
                    continue

                # DuckDB's actual data is checked as well as its
                # schema. This catches nullable data even when
                # schema metadata does not explicitly describe it.
                nullable = (
                    str(column_info.get("null", "")).upper()
                    == "YES"
                )

                if nullable or has_nulls(table, column):
                    if has_nulls(table, column):
                        warn(
                            "NULL_FILTER",
                            "warning",
                            "WHERE predicate filters a column containing "
                            "NULL values. SQL comparisons evaluate to "
                            "UNKNOWN for NULL, so those rows are excluded.",
                            table=table,
                            column=column,
                            alias=alias,
                            predicate=predicate.group(0).strip()
                        )

    # ---------------------------------------------------------
    # 3. Detect aggregates exposed to fan-out.
    # ---------------------------------------------------------

    has_aggregate = bool(re.search(
        r"\b(SUM|AVG|COUNT|MIN|MAX)\s*\(",
        sql,
        re.IGNORECASE
    ))

    if has_aggregate and fanout_detected:
        warn(
            "AGGREGATE_FANOUT",
            "error",
            "An aggregate is calculated over a result containing "
            "a potentially multiplying join. Measures may be "
            "overstated.",
            aggregates=re.findall(
                r"\b(SUM|AVG|COUNT|MIN|MAX)\s*\(",
                sql,
                re.IGNORECASE
            ),
            recommendation=(
                "Aggregate at the intended grain before joining, "
                "or verify the join cardinality."
            )
        )

    return warnings
```

## 2. Why these checks work

### Fan-out detection without executing the full join twice

The reviewer checks the join-side key directly:

sql

```sql
SELECT customer_id, COUNT(*)
FROM orders
GROUP BY customer_id
HAVING COUNT(*) > 1;
```

If `customers.customer_id` is unique but `orders.customer_id` has multiple matching rows, a customer-level measure can be repeated after the join.

This is a pre-join cardinality check. It avoids executing the complete joined SELECT a second time. It identifies a potential fan-out rather than claiming every duplicate necessarily contributes to the final filtered result.

There are two distinct row-count problems:

- Multiplication: multiple matching rows create more output rows.
- Row loss: an INNER JOIN excludes records without a match.

The implementation checks both for the supported equality-join pattern.

### Why NULL filtering is a correctness issue

SQL uses three-valued logic:

| Expression | Result |
| --- | --- |
| `100 > 50` | TRUE |
| `10 > 50` | FALSE |
| `NULL > 50` | UNKNOWN |

A `WHERE` clause retains only TRUE rows.

Consequently:

sql

```sql
WHERE orders.amount > 50
```

silently removes records where `amount IS NULL`.

That is potentially a business correctness issue: unknown revenue is not necessarily zero revenue, and excluding it can change reporting populations.

If the intended rule is to retain unknown values, the predicate needs to express that:

sql

```sql
WHERE orders.amount > 50
   OR orders.amount IS NULL
```

The correct treatment depends on the business definition.

## 3. Important implementation qualification

The code above is a lightweight SQL reviewer, not a complete SQL parser. In particular, its join detection assumes the joined table is the right-hand side of the equality predicate.

For production use, I would tighten that assumption. In the join extraction section, identify the newly joined alias from the JOIN clause and select its corresponding equality key, regardless of which side of `=` it appears on.

Also, the current implementation detects duplicate keys in the underlying table, not necessarily duplicates surviving all query filters. That makes it a useful conservative warning, not proof of an inflated result.

## 4. Testing

Here is a minimal DuckDB fixture using your table names:

python

```python
import duckdb

con = duckdb.connect(":memory:")

con.execute("""
CREATE TABLE customers (
    customer_id INTEGER,
    customer_name VARCHAR
);

CREATE TABLE orders (
    order_id INTEGER,
    customer_id INTEGER,
    amount DECIMAL(10, 2)
);

INSERT INTO customers VALUES
    (1, 'Alice'),
    (2, 'Bob'),
    (3, 'Carol');

INSERT INTO orders VALUES
    (101, 1, 100),
    (102, 1, 100),
    (103, 2, 200),
    (104, 3, NULL);
""")

sql = """
SELECT
    c.customer_id,
    SUM(o.amount) AS revenue
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
WHERE o.amount > 50
GROUP BY c.customer_id
"""

result = review_query(con, sql)

for warning in result:
    print(warning)
```

Expected findings:

| Warning | Expected |
| --- | --- |
| `JOIN_FANOUT` | Yes: customer 1 has two orders |
| `NULL_FILTER` | Yes: order 104 has NULL amount |
| `AGGREGATE_FANOUT` | Yes: SUM is calculated over a potentially multiplied result |

Note that this fixture demonstrates the warnings; it does not establish that the $1,240 incident had the same cause.

## 5. Evaluating your Phase 1 queries

Not yet verifiedI don't have the Phase 1 queries or their underlying data, so I cannot honestly say whether this reviewer would have caught the $1,240.

Please paste:

1. Phase 1 query #1.
2. Phase 1 query #2.
3. The relevant table definitions for `customers` and `orders`, if available.

I'll evaluate each against the three warning categories and determine:

- Whether the join multiplied or dropped rows.
- Whether nullable filters excluded records.
- Whether an aggregate was calculated at the wrong grain.
- Whether the resulting discrepancy could explain the $1,240.

The ultimate test: Would the reviewer have raised a warning before the dashboard's incorrect number reached the board? That's the standard I'd use to judge this implementation.
```

```
n
```
**Reply:**

[paste the reply exactly as it came back; put any code inside a ```python fence]

**Changes I made:**
- [one line per change]

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.